In [1]:
# %%
# =============================================================================
# NOTEBOOK: 02_baseline_D_TDC.ipynb
# Paper D (TDC) — baselines before text-strategy experiments
#
# Establishes the reference points every text strategy must beat:
#   (1) NUMERIC baseline — financial ratios only (logistic reg + gradient boost)
#   (2) TEXT-LENGTH baseline — can mere document length predict bankruptcy?
#       (critical confound: bankrupt firms have ~1.5x longer MD&A)
#   (3) combined numeric + length — the real bar for "text CONTENT adds value"
#
# Small, balanced data (train 111 / test 111, ~50% bankrupt) → report accuracy,
# ROC-AUC, F1 with repeated CV + bootstrap CIs (single split is too noisy).
#
# Data: text-based bankruptcy dataset, Mendeley DOI 10.17632/stf3kg7fw3
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
TEXT_DIR = ROOT / "data" / "raw" / "text_bankruptcy"
TAB = ROOT / "artifacts" / "tables"
TAB.mkdir(parents=True, exist_ok=True)


def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name


# --- Load and combine train+test (we'll use repeated CV on the pooled set) ---
Xtr = pd.read_csv(TEXT_DIR / "NUM10K_X_train_2021June.csv")
Xte = pd.read_csv(TEXT_DIR / "NUM10K_X_test_2021June.csv")
ytr = pd.read_csv(TEXT_DIR / "NUM10K_y_train_2021June.csv")["Bankruptcy"]
yte = pd.read_csv(TEXT_DIR / "NUM10K_y_test_2021June.csv")["Bankruptcy"]
Xtr["Bankruptcy"] = ytr.values; Xte["Bankruptcy"] = yte.values
df = pd.concat([Xtr, Xte], ignore_index=True)
print(f"Pooled: {len(df)} firms, {int(df.Bankruptcy.sum())} bankrupt "
      f"({df.Bankruptcy.mean():.1%})")

y = df["Bankruptcy"].values
num_cols = [c for c in df.columns
            if c not in ("cik", "RiskFactors", "MDandA", "date", "Bankruptcy")]
# text length features (the confound)
df["_mdna_wc"] = df["MDandA"].fillna("").astype(str).str.split().str.len()
df["_rf_wc"] = df["RiskFactors"].fillna("").astype(str).str.split().str.len()

# --- Evaluation helper: repeated stratified CV with bootstrap-style spread ----
def eval_features(name, Xmat):
    Xmat = np.nan_to_num(Xmat.astype(float))
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
    out = {}
    for mdl_name, mdl in [
        ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
        ("gbm", GradientBoostingClassifier(random_state=42)),
    ]:
        auc = cross_val_score(mdl, Xmat, y, cv=cv, scoring="roc_auc")
        acc = cross_val_score(mdl, Xmat, y, cv=cv, scoring="accuracy")
        out[mdl_name] = (auc.mean(), auc.std(), acc.mean())
    # report the better model
    best = max(out, key=lambda k: out[k][0])
    m = out[best]
    print(f"  {name:28s} [{best}]: ROC-AUC {m[0]:.3f} ±{m[1]:.3f}, acc {m[2]:.3f}")
    return {"features": name, "model": best, "roc_auc": m[0],
            "roc_auc_std": m[1], "accuracy": m[2]}


print("\n── Baselines (repeated 5×10 stratified CV) ──")
rows = []
rows.append(eval_features("numeric only (39)", df[num_cols].values))
rows.append(eval_features("text-length only (2)", df[["_mdna_wc", "_rf_wc"]].values))
rows.append(eval_features("numeric + length", df[num_cols + ["_mdna_wc", "_rf_wc"]].values))

res = pd.DataFrame(rows)
res.round(4).to_csv(TAB / "D_baselines.csv", index=False)
print(f"\n  → saved: {rel(TAB / 'D_baselines.csv')}")
print("\nRead:")
print("  • numeric-only = the financial-statement baseline")
print("  • text-length-only = how much the CONFOUND (length) alone predicts")
print("  • Any text-CONTENT strategy must beat 'numeric + length' to prove the")
print("    content (not just length) carries signal.")

Pooled: 222 firms, 111 bankrupt (50.0%)

── Baselines (repeated 5×10 stratified CV) ──
  numeric only (39)            [gbm]: ROC-AUC 0.862 ±0.050, acc 0.800
  text-length only (2)         [logreg]: ROC-AUC 0.577 ±0.073, acc 0.543
  numeric + length             [gbm]: ROC-AUC 0.865 ±0.050, acc 0.802

  → saved: artifacts\tables\D_baselines.csv

Read:
  • numeric-only = the financial-statement baseline
  • text-length-only = how much the CONFOUND (length) alone predicts
  • Any text-CONTENT strategy must beat 'numeric + length' to prove the
    content (not just length) carries signal.
